# Module 8A: DQ Dashboard - Snowflake Native Dashboards + Power BI

## Learning Objectives
- Create BI-ready DQ reporting views
- Build a dashboard using **Snowflake Dashboards** (SQL query tiles)
- Understand Power BI best practices for DQ views

> **Business Value:** Native dashboards = zero additional infrastructure. Same platform that runs DQ checks also displays results.

---
> **Role:** `CORP_DQ_ADMIN` | **Time:** ~45 minutes | **Variant:** Native + Power BI

> **What this does:** Sets your session context to the lab role, database, and warehouse.


In [ ]:
USE ROLE CORP_DQ_ADMIN;
USE DATABASE CORP_DWH;
USE WAREHOUSE COMPUTE_WH;

---
## Shared Setup: Create DQ Reporting Views

> **Business Value:** BI tools cannot call table functions directly. Views provide a stable, queryable interface.

In [ ]:
CREATE OR REPLACE VIEW CORP_DWH.DQ.V_DQ_RESULTS_FLAT AS
SELECT
    r.REF_ENTITY_NAME AS TABLE_NAME, r.METRIC_NAME,
    r.ARGUMENT_NAMES AS COLUMN_CHECKED, r.VALUE AS METRIC_VALUE,
    r.EXPECTATION_NAME, r.EXPECTATION_RESULT, r.MEASUREMENT_TIME,
    COALESCE(c.SEVERITY, 'MEDIUM') AS SEVERITY,
    COALESCE(c.OWNER, 'Unassigned') AS RULE_OWNER,
    COALESCE(c.RULE_TYPE, 'SYSTEM') AS RULE_TYPE,
    CASE WHEN r.EXPECTATION_RESULT = 'MET' THEN 'PASS'
         WHEN r.EXPECTATION_RESULT = 'NOT_MET' THEN 'FAIL'
         ELSE 'NO_EXPECTATION' END AS STATUS
FROM TABLE(SNOWFLAKE.LOCAL.DATA_QUALITY_MONITORING_RESULTS(
    REF_ENTITY_NAME => 'CORP_DWH.GOLD.DIM_CUSTOMER', REF_ENTITY_DOMAIN => 'TABLE'
)) r
LEFT JOIN CORP_DWH.DQ.RULES_CATALOG c
    ON UPPER(r.METRIC_NAME) LIKE '%' || REPLACE(UPPER(c.RULE_NAME), ' ', '_') || '%';

> **What this does:** Creates V_DQ_SCORECARD view that calculates per-table health scores from the latest DQ expectation results.


In [ ]:
CREATE OR REPLACE VIEW CORP_DWH.DQ.V_DQ_SCORECARD AS
WITH latest AS (
    SELECT 'CORP_DWH.GOLD.DIM_CUSTOMER' AS TABLE_NAME,
        METRIC_NAME, ARGUMENT_NAMES, VALUE, EXPECTATION_NAME, EXPECTATION_RESULT, MEASUREMENT_TIME,
        ROW_NUMBER() OVER (PARTITION BY METRIC_NAME, ARGUMENT_NAMES ORDER BY MEASUREMENT_TIME DESC) AS RN
    FROM TABLE(SNOWFLAKE.LOCAL.DATA_QUALITY_MONITORING_RESULTS(
        REF_ENTITY_NAME => 'CORP_DWH.GOLD.DIM_CUSTOMER', REF_ENTITY_DOMAIN => 'TABLE'))
    WHERE EXPECTATION_NAME IS NOT NULL
)
SELECT TABLE_NAME,
    COUNT(*) AS TOTAL_EXPECTATIONS,
    COUNT(CASE WHEN EXPECTATION_RESULT = 'MET' THEN 1 END) AS PASSED,
    COUNT(CASE WHEN EXPECTATION_RESULT = 'NOT_MET' THEN 1 END) AS FAILED,
    ROUND(100.0 * COUNT(CASE WHEN EXPECTATION_RESULT = 'MET' THEN 1 END) / NULLIF(COUNT(*), 0), 1) AS HEALTH_SCORE_PCT,
    MAX(MEASUREMENT_TIME) AS LAST_EVALUATED
FROM latest WHERE RN = 1 GROUP BY TABLE_NAME;

> **What this does:** Creates V_DQ_TREND view that provides hourly time-series data for charting quality metrics over time.


In [ ]:
CREATE OR REPLACE VIEW CORP_DWH.DQ.V_DQ_TREND AS
SELECT DATE_TRUNC('HOUR', MEASUREMENT_TIME) AS MEASUREMENT_HOUR,
    METRIC_NAME, VALUE AS METRIC_VALUE, EXPECTATION_RESULT, MEASUREMENT_TIME
FROM TABLE(SNOWFLAKE.LOCAL.DATA_QUALITY_MONITORING_RESULTS(
    REF_ENTITY_NAME => 'CORP_DWH.GOLD.DIM_CUSTOMER', REF_ENTITY_DOMAIN => 'TABLE'))
WHERE EXPECTATION_NAME IS NOT NULL ORDER BY MEASUREMENT_TIME DESC;

> **What this does:** Creates V_DQ_EXECUTIVE_SUMMARY view that aggregates all checks into a single overall health percentage.


In [ ]:
CREATE OR REPLACE VIEW CORP_DWH.DQ.V_DQ_EXECUTIVE_SUMMARY AS
SELECT 'CORP_DWH' AS DATA_ESTATE, COUNT(*) AS TOTAL_CHECKS,
    COUNT(CASE WHEN EXPECTATION_RESULT = 'MET' THEN 1 END) AS CHECKS_PASSING,
    COUNT(CASE WHEN EXPECTATION_RESULT = 'NOT_MET' THEN 1 END) AS CHECKS_FAILING,
    ROUND(100.0 * COUNT(CASE WHEN EXPECTATION_RESULT = 'MET' THEN 1 END) / NULLIF(COUNT(*), 0), 1) AS OVERALL_HEALTH_PCT,
    CURRENT_TIMESTAMP() AS AS_OF
FROM TABLE(SNOWFLAKE.LOCAL.DATA_QUALITY_MONITORING_RESULTS(
    REF_ENTITY_NAME => 'CORP_DWH.GOLD.DIM_CUSTOMER', REF_ENTITY_DOMAIN => 'TABLE'))
WHERE EXPECTATION_NAME IS NOT NULL;

---
## Snowflake Dashboard Tiles

Go to **Projects > Dashboards > + Dashboard** and add these SQL tiles:

### Tile 1: Health KPI (Scorecard)

In [ ]:
SELECT OVERALL_HEALTH_PCT AS "Data Quality Health (%)" FROM CORP_DWH.DQ.V_DQ_EXECUTIVE_SUMMARY;

### Tile 2: Pass vs Fail (Bar Chart)

In [ ]:
SELECT STATUS, COUNT(*) AS CHECK_COUNT
FROM CORP_DWH.DQ.V_DQ_RESULTS_FLAT
WHERE STATUS IN ('PASS', 'FAIL')
GROUP BY STATUS;

### Tile 3: Failures by Severity (Horizontal Bar)

In [ ]:
SELECT SEVERITY, COUNT(*) AS FAILURE_COUNT
FROM CORP_DWH.DQ.V_DQ_RESULTS_FLAT
WHERE STATUS = 'FAIL'
GROUP BY SEVERITY
ORDER BY CASE SEVERITY WHEN 'CRITICAL' THEN 1 WHEN 'HIGH' THEN 2 WHEN 'MEDIUM' THEN 3 ELSE 4 END;

### Tile 4: Top Failing Rules (Table)

In [ ]:
SELECT METRIC_NAME AS RULE, COLUMN_CHECKED, METRIC_VALUE AS VIOLATIONS, SEVERITY, RULE_OWNER
FROM CORP_DWH.DQ.V_DQ_RESULTS_FLAT
WHERE STATUS = 'FAIL' AND METRIC_VALUE > 0
ORDER BY METRIC_VALUE DESC LIMIT 10;

### Tile 5: Trend (Line Chart)

In [ ]:
SELECT MEASUREMENT_HOUR,
    COUNT(CASE WHEN EXPECTATION_RESULT = 'MET' THEN 1 END) AS PASSING,
    COUNT(CASE WHEN EXPECTATION_RESULT = 'NOT_MET' THEN 1 END) AS FAILING
FROM CORP_DWH.DQ.V_DQ_TREND
GROUP BY MEASUREMENT_HOUR ORDER BY MEASUREMENT_HOUR;

---
## Power BI Best Practices

### Connection
- Use **Snowflake connector** in Power BI Desktop
- **DirectQuery** for V_DQ_SCORECARD + V_DQ_EXECUTIVE_SUMMARY (small, always fresh)
- **Import** for V_DQ_TREND (historical, refresh hourly)
- Role: `CORP_DQ_STEWARD`

### Recommended Visuals
| Visual | Source | Purpose |
|--------|--------|---------|
| Card | V_DQ_EXECUTIVE_SUMMARY.OVERALL_HEALTH_PCT | KPI "87% Healthy" |
| Gauge | V_DQ_SCORECARD.HEALTH_SCORE_PCT | Per-table health |
| Stacked Bar | V_DQ_RESULTS_FLAT by SEVERITY | Pass/Fail breakdown |
| Line Chart | V_DQ_TREND by MEASUREMENT_HOUR | Health over time |
| Matrix | V_DQ_RESULTS_FLAT (TABLE x METRIC) | Heatmap |

### DAX Measures
```dax
Health Score = DIVIDE(
    COUNTROWS(FILTER('V_DQ_RESULTS_FLAT', [STATUS] = "PASS")),
    COUNTROWS('V_DQ_RESULTS_FLAT')) * 100

Critical Failures = COUNTROWS(FILTER('V_DQ_RESULTS_FLAT', 
    [STATUS] = "FAIL" && [SEVERITY] = "CRITICAL"))
```

### Row-Level Security
Map Power BI roles to `RULE_OWNER` column so each team sees only their metrics.

---
## Checkpoint

> **What this does:** Verifies your work so far. All checks should show [PASS].


In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
print("=" * 50)
print("CHECKPOINT: Dashboard Views Ready")
print("=" * 50)
for v in ['V_DQ_RESULTS_FLAT', 'V_DQ_SCORECARD', 'V_DQ_TREND', 'V_DQ_EXECUTIVE_SUMMARY']:
    try:
        cnt = session.sql(f"SELECT COUNT(*) AS C FROM CORP_DWH.DQ.{v}").collect()[0]['C']
        print(f"  [PASS] {v} -- {cnt} rows")
    except Exception as e:
        print(f"  [FAIL] {v}: {str(e)[:50]}")
print("=" * 50)

---
**Next:** Try 8B (Python charts) or 8C (Streamlit), or proceed to `9_TEARDOWN`.